In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [15]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [16]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model_provider="openai",
    base_url=os.environ.get("OPENAI_BASE_URL"),
    api_key=os.environ.get("OPENAI_API_KEY"),
    model=os.environ.get("OPENAI_MODEL"),
    temperature=0.3,
)

agent = create_agent(
    model=model,
    tools=[sql_query]
)

In [18]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [19]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='97c2e3fa-7d40-45e7-aa40-7770dfdb0230'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 288, 'total_tokens': 341, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen3.5-397B-A17B-FP8', 'system_fingerprint': None, 'id': 'chatcmpl-ac5e6c276087e9ca', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d29c4-abf2-7623-95be-d17ddca42868-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT artist, COUNT(*) as song_count FROM songs WHERE artist LIKE 'S%' GROUP BY artist ORDER BY song_count DESC LIMIT 1;"}, 'id': 'chatcmpl-tool-ae2755b0bb340708', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 288, 'output_tokens': 53, 'total_tokens': 341, 'input_token_

In [20]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT Artist.Name, COUNT(Track.TrackId) as TrackCount FROM Artist JOIN Album ON Artist.ArtistId = Album.ArtistId JOIN Track ON Album.AlbumId = Track.AlbumId WHERE Artist.Name LIKE 'S%' GROUP BY Artist.Name ORDER BY TrackCount DESC LIMIT 1;
